In [12]:
import os
import numpy as np
from nilearn import image
import numpy as np
import matplotlib.pyplot as plt

In [2]:
fcontrast_dir = "/Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/21styear/"
left_mask_path = "TPJ_refined_mask_L.nii.gz"
right_mask_path = "TPJ_refined_mask_R.nii.gz"


In [15]:
# ----------------------------
# Load 21styear data
# ----------------------------



In [5]:
# ----------------------------
# Load original masks
# ----------------------------
left_mask_orig = image.load_img(left_mask_path)
right_mask_orig = image.load_img(right_mask_path)

# ----------------------------
# Collect all .nii.gz files
# ----------------------------
img_files = [f for f in os.listdir(fcontrast_dir) if f.endswith(".nii.gz")]
img_files.sort()
n_imgs = len(img_files)
print(f"Found {n_imgs} images")

# initialize timelines
first_img = image.load_img(os.path.join(fcontrast_dir, img_files[0]))

# Resample masks to the first image space
left_mask_resampled = image.resample_to_img(left_mask_orig, first_img, interpolation='nearest')
right_mask_resampled = image.resample_to_img(right_mask_orig, first_img, interpolation='nearest')

left_mask_data = left_mask_resampled.get_fdata().astype(bool)
right_mask_data = right_mask_resampled.get_fdata().astype(bool)

n_left_voxels = np.sum(left_mask_data)
n_right_voxels = np.sum(right_mask_data)

left_timeline = np.zeros((n_imgs, n_left_voxels))
right_timeline = np.zeros((n_imgs, n_right_voxels))

# ----------------------------
# Loop through images
# ----------------------------
for i, fname in enumerate(img_files):
    img_path = os.path.join(fcontrast_dir, fname)
    img = image.load_img(img_path)
    
    # Resample masks to current image in case of slightly different shapes
    left_mask_res = image.resample_to_img(left_mask_orig, img, interpolation='nearest', copy_header=True, force_resample=True)
    right_mask_res = image.resample_to_img(right_mask_orig, img, interpolation='nearest', copy_header=True, force_resample=True)
    
    left_mask_data = left_mask_res.get_fdata().astype(bool)
    right_mask_data = right_mask_res.get_fdata().astype(bool)
    
    img_data = img.get_fdata()
    
    # Extract voxel values
    left_timeline[i, :] = img_data[left_mask_data]
    right_timeline[i, :] = img_data[right_mask_data]

print("Extraction complete")
print("Left TPJ timeline shape:", left_timeline.shape)
print("Right TPJ timeline shape:", right_timeline.shape)

# ----------------------------
# Save timelines
# ----------------------------
np.save("left_TPJ_timeline.npy", left_timeline)
np.save("right_TPJ_timeline.npy", right_timeline)
print("Saved left/right TPJ timelines as .npy files")


Found 25 images


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15391/2974378550.py:30: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  left_mask_resampled = image.resample_to_img(left_mask_orig, first_img, interpolation='nearest')
/opt/anaconda3/envs/renv/lib/python3.12/site-packages/nilearn/image/resampling.py:805: FutureWarning: From release 0.13.0 onwards, this function will, by default, copy the header of the input image to the output. Currently, the header is reset to the default Nifti1Header. To suppress this warning and use the new behavior, set `copy_header=True`.
  return resample_img(
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15391/2974378550.py:31: FutureWarning: 'force_resample' will be set to 'True' by default in Nilearn 0.13.0.
Use 'force_resample=True' to suppress this warning.
  right_mask_resampled = image.resample_to_img(right_mask_orig, first_img, interpolation='nea

Extraction complete
Left TPJ timeline shape: (25, 239)
Right TPJ timeline shape: (25, 144)
Saved left/right TPJ timelines as .npy files
